# Wikipedia Company Profile APP

Take the company name as input. Extract the below company related details from Wikipedia:


*   The founder of the company.
*   When it was founded.
*   The current market capital of the company.
*   How many employees are working in it.
*   A brief 4-line summary of the company.

---
**Sample Output Below**
   ```python
 {
    "Company_Name": "Tata Consultancy Services Limited",
    "Founder": "Tata Sons Limited",
    "Start_Date": "1968",
    "Revenue": 206858.05,
    "Employees": "601,546",
    "Summary": "TCS is an Indian multinational information technology (IT) services and consulting company headquartered in Mumbai. It is a part of the Tata Group and operates in 150 locations across 46 countries."
}
```

**Follow these detailed instructions.**


# 1 - Install Required Packages
   Install necessary Python packages using pip, including `openai`, `langchain`, `langchain-cohere`, `langchain-community`, and `wikipedia`.


In [1]:
!pip install langchain_openai
!pip install langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.9 MB/s eta 0:00:00


In [2]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [3]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=e4b9466a7a2ffd24cb7b15ccd9530fdae5f282a36634cef211d1248c9be6417d
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia



# 2 - Set Up Environment Variables for API Keys:
   Set up your OpenAI and Cohere API keys as environment variables. This can be done securely using Google Colab's `userdata`.


In [4]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OpenAI_API_key')

# 3 - Initialize Language Models and Chains:
   Import the necessary modules from LangChain and initialize the language models for OpenAI and Cohere.


In [26]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import WikipediaLoader

# 4 - Load Company Data from Wikipedia:
   Use the `WikipediaLoader` from LangChain to fetch company information from Wikipedia.


In [27]:
loder = WikipediaLoader(query = " Tata Consultancy Services Limited", load_max_docs=1)

In [28]:
input_data= loder.load()

# 5 - Define Output Parsing Schema:
   Use Pydantic to define the schema for the desired output and create a custom output parser.


In [29]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [30]:
class wikireport(BaseModel):
  Company_Name: str = Field(description = "The name of the company")
  Founder: str = Field(description = "The founder of the company")
  Start_Date: str = Field(description = "When it was founded")
  Revenue: float = Field(description = "The current market capital of the company")
  Employees: str = Field(description = "How many employees are working in it")
  Summary: str = Field(description = "A brief 4-line summary of the company")

custom_parser = PydanticOutputParser(pydantic_object = wikireport)

# 6 - Create a Prompt Template:
   Define a prompt template to structure the input for the language model.


In [31]:
template = """
take the input data {input_data} as input , and give output as expected
{format_instruction}
"""
prompt = PromptTemplate(
    input_variables = ["input_data" , "format_instruction"],
    template = template )
llm = ChatOpenAI(temperature = 0, model= "gpt-4o-mini")
Chain = prompt | llm



In [38]:
result = Chain.invoke({"input_data": input_data,
             "format_instruction" : custom_parser.get_format_instructions() }
)

In [22]:
print(result.content)

```json
{
  "Company_Name": "Tata Consultancy Services",
  "Founder": "Tata Sons Limited",
  "Start_Date": "1968",
  "Revenue": 200000000000,
  "Employees": "Over 500,000",
  "Summary": "Tata Consultancy Services Limited (TCS) is an Indian multinational technology company specializing in information technology services and consulting. Headquartered in Mumbai, it is a part of the Tata Group and operates in 150 locations across 46 countries. As of 2024, Tata Sons owned 71.74% of TCS, and close to 80% of Tata Sons' dividend income came from TCS."
}
```


#  7-Initialize the Language Model Chain:
   Set up the language model chain using the defined prompt and language model (OpenAI or Cohere).

#  8 - Invoke the Chain and Fetch Results:
   Invoke the chain with the company documents and format instructions, then print the results.